In [3]:
import pickle
import numpy as np
import tensorflow as tf
from sklearn.utils import class_weight
from functions import prepare_datasets, slice_features, build_dynamic_model, Config

def train_and_save_resources():
    print("--- 1. CONFIGURING DATASET (6 CLASSES) ---")
    # Explicitly defining the 6 classes (Excluding Surprise)
    TARGET_CLASSES = ['neutral', 'sadness', 'joy', 'anger', 'fear', 'disgust']
    print(f"Target Classes: {TARGET_CLASSES}")
    
    # Load Data
    data, scaler = prepare_datasets(TARGET_CLASSES, augment=False)
    
    # Slice Features
    feats = ['MFCC', 'Chroma', 'Spectral_Contrast', 'Zero_Crossing_Rate', 'RMS_Energy', 'Sentiment']
    inputs, shapes, types = slice_features(data, feats)
    
    print("\n--- 2. TRAINING FINAL MODEL ---")
    # Build Model
    tf.keras.backend.clear_session()
    model = build_dynamic_model(types, shapes, len(TARGET_CLASSES), data['num_sent_classes'])
    
    # Compute Class Weights (Vital for Fear/Disgust)
    cw = class_weight.compute_class_weight('balanced', classes=np.unique(data['y_train']), y=data['y_train'])
    
    # Train
    model.fit(inputs['train'], data['y_train'], 
              validation_data=(inputs['dev'], data['y_dev']),
              epochs=20, # Train sufficiently
              batch_size=32, 
              class_weight=dict(enumerate(cw)),
              verbose=1)
    
    print("\n--- 3. SAVING RESOURCES ---")
    # Save Model
    model.save('model_bimodal_6class.keras')
    print("✅ Model saved as: 'model_bimodal_6class.keras'")
    
    # Save Label Encoder
    with open('label_encoder_6class.pkl', 'wb') as f:
        pickle.dump(data['le_target'], f)
    print("✅ Label Encoder saved as: 'label_encoder_6class.pkl'")

    # 3. SAVE THE SCALER
    with open('scaler_6class.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    print("✅ Scaler saved as: 'scaler_6class.pkl'")

if __name__ == "__main__":
    train_and_save_resources()

--- 1. CONFIGURING DATASET (6 CLASSES) ---
Target Classes: ['neutral', 'sadness', 'joy', 'anger', 'fear', 'disgust']

--- Preparing Data for Classes: ['neutral', 'sadness', 'joy', 'anger', 'fear', 'disgust'] ---
Processing 8783 samples from dataset_meld/prepared_train_sent_emo.csv (Augment=False)...


100%|██████████| 8783/8783 [05:40<00:00, 25.82it/s]


Processing 958 samples from dataset_meld/prepared_dev_sent_emo.csv (Augment=False)...


100%|██████████| 958/958 [00:31<00:00, 30.47it/s]


Processing 2329 samples from dataset_meld/prepared_test_sent_emo.csv (Augment=False)...


100%|██████████| 2329/2329 [01:20<00:00, 29.07it/s]



--- 2. TRAINING FINAL MODEL ---



Epoch 1/20

275/275 [==============================] - 13s 26ms/step - loss: 1.4594 - accuracy: 0.5910 - val_loss: 0.4931 - val_accuracy: 0.7829
Epoch 2/20
275/275 [==============================] - 6s 20ms/step - loss: 0.9784 - accuracy: 0.8159 - val_loss: 0.4601 - val_accuracy: 0.7797
Epoch 3/20
275/275 [==============================] - 6s 22ms/step - loss: 0.9237 - accuracy: 0.8202 - val_loss: 0.4649 - val_accuracy: 0.7766
Epoch 4/20
275/275 [==============================] - 8s 28ms/step - loss: 0.9028 - accuracy: 0.8282 - val_loss: 0.4578 - val_accuracy: 0.7850
Epoch 5/20
275/275 [==============================] - 6s 23ms/step - loss: 0.9010 - accuracy: 0.8296 - val_loss: 0.4677 - val_accuracy: 0.7630
Epoch 6/20
275/275 [==============================] - 5s 19ms/step - loss: 0.8862 - accuracy: 0.8305 - val_loss: 0.4791 - val_accuracy: 0.7641
Epoch 7/20
275/275 [==============================] - 5s 19ms/step - loss: 0.8738 - accuracy: 0.8359 - v